# 03 — Modeling: Flight Delay Prediction

Train and evaluate classification models to predict `ArrDel15` (arrival delay >= 15 min).

**Approach:**
1. Establish a baseline (historical frequency by weekday + dest airport)
2. Train Logistic Regression, Random Forest, and Gradient Boosting
3. Compare all models on held-out test set
4. Export the best model for potential app integration

**Rules:**
- Target: `ArrDel15` (ADR-0001)
- Cancelled flights already excluded in feature engineering (ADR-0002)
- Random seed: 42

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, confusion_matrix,
    RocCurveDisplay
)
import joblib
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

## 1. Load prepared data

In [ ]:
df = pd.read_csv('../data/flights_prepared.csv')
print(f'Dataset: {len(df):,} rows')
print(f'Delay rate: {df["ArrDel15"].mean():.3f}')
df.head()

## 2. Define features and split

In [ ]:
num_features = [
    'Month', 'DayOfWeek', 'DepHour',
    'carrier_delay_rate', 'origin_delay_rate', 'dest_delay_rate',
    'route_delay_rate', 'route_flights'
]
cat_features = ['TimeBucket', 'Season']
target = 'ArrDel15'

X = df[num_features + cat_features]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED, stratify=y
)

print(f'Train: {len(X_train):,} | Test: {len(X_test):,}')
print(f'Train delay rate: {y_train.mean():.3f}')
print(f'Test  delay rate: {y_test.mean():.3f}')

## 3. Preprocessing pipeline

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_features),
        ('cat', OneHotEncoder(drop='first', handle_unknown='infrequent_if_exist'), cat_features),
    ]
)

## 4. Baseline: Historical frequency

The current web app uses historical frequency by (DayOfWeek, DestAirportID). Simulate this as a baseline.

In [ ]:
# Build baseline lookup from training data
df_train = df.iloc[X_train.index].copy()
baseline_lookup = df_train.groupby(['DayOfWeek', 'DestAirportID'])['ArrDel15'].mean().to_dict()
overall_rate = df_train['ArrDel15'].mean()

# Predict on test set using the lookup
df_test = df.iloc[X_test.index].copy()
y_baseline_prob = df_test.apply(
    lambda r: baseline_lookup.get((r['DayOfWeek'], r['DestAirportID']), overall_rate), axis=1
)
y_baseline_pred = (y_baseline_prob >= 0.5).astype(int)

baseline_auc = roc_auc_score(y_test, y_baseline_prob)
baseline_acc = accuracy_score(y_test, y_baseline_pred)
print(f'Baseline AUC:      {baseline_auc:.4f}')
print(f'Baseline Accuracy: {baseline_acc:.4f}')

## 5. Model training

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(
        max_iter=1000, random_state=RANDOM_SEED, class_weight='balanced'
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=200, max_depth=10, random_state=RANDOM_SEED,
        class_weight='balanced', n_jobs=-1
    ),
    'Gradient Boosting': GradientBoostingClassifier(
        n_estimators=200, max_depth=5, learning_rate=0.1,
        random_state=RANDOM_SEED
    ),
}

results = {}
trained_pipelines = {}

for name, model in models.items():
    print(f'\n--- {name} ---')
    pipe = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', model)
    ])
    pipe.fit(X_train, y_train)
    
    y_pred = pipe.predict(X_test)
    y_prob = pipe.predict_proba(X_test)[:, 1]
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)
    
    results[name] = {'Accuracy': acc, 'Precision': prec, 'Recall': rec, 'F1': f1, 'AUC': auc}
    trained_pipelines[name] = pipe
    
    print(f'Accuracy:  {acc:.4f}')
    print(f'Precision: {prec:.4f}')
    print(f'Recall:    {rec:.4f}')
    print(f'F1:        {f1:.4f}')
    print(f'AUC:       {auc:.4f}')

## 6. Results comparison

In [ ]:
# Add baseline to results
results['Baseline (historical freq)'] = {
    'Accuracy': baseline_acc,
    'Precision': precision_score(y_test, y_baseline_pred),
    'Recall': recall_score(y_test, y_baseline_pred),
    'F1': f1_score(y_test, y_baseline_pred),
    'AUC': baseline_auc
}

results_df = pd.DataFrame(results).T.round(4)
results_df = results_df.sort_values('AUC', ascending=False)
print(results_df.to_string())

In [ ]:
# Visual comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# AUC comparison
colors = ['#F44336' if 'Baseline' in n else '#5B8DEF' for n in results_df.index]
axes[0].barh(results_df.index, results_df['AUC'], color=colors)
axes[0].set_xlabel('AUC')
axes[0].set_title('Model comparison: AUC')
for i, v in enumerate(results_df['AUC']):
    axes[0].text(v + 0.002, i, f'{v:.4f}', va='center')

# F1 comparison
axes[1].barh(results_df.index, results_df['F1'], color=colors)
axes[1].set_xlabel('F1 Score')
axes[1].set_title('Model comparison: F1')
for i, v in enumerate(results_df['F1']):
    axes[1].text(v + 0.002, i, f'{v:.4f}', va='center')

plt.tight_layout()
plt.show()

## 7. ROC curves

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

for name, pipe in trained_pipelines.items():
    RocCurveDisplay.from_estimator(pipe, X_test, y_test, name=name, ax=ax)

# Plot baseline ROC
from sklearn.metrics import roc_curve
fpr_b, tpr_b, _ = roc_curve(y_test, y_baseline_prob)
ax.plot(fpr_b, tpr_b, '--', label=f'Baseline (AUC={baseline_auc:.4f})', color='gray')

ax.plot([0, 1], [0, 1], 'k--', alpha=0.3, label='Random')
ax.set_title('ROC Curves')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

## 8. Best model: confusion matrix

In [ ]:
# Pick best model by AUC
best_name = results_df.drop('Baseline (historical freq)', errors='ignore').idxmax()['AUC']
best_pipe = trained_pipelines[best_name]
y_best_pred = best_pipe.predict(X_test)

print(f'Best model: {best_name}\n')
print(classification_report(y_test, y_best_pred, target_names=['On time', 'Delayed']))

cm = confusion_matrix(y_test, y_best_pred)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['On time', 'Delayed'], yticklabels=['On time', 'Delayed'])
ax.set_ylabel('Actual')
ax.set_xlabel('Predicted')
ax.set_title(f'Confusion Matrix: {best_name}')
plt.tight_layout()
plt.show()

## 9. Feature importance (best model)

In [ ]:
# Get feature names after preprocessing
ohe_features = list(best_pipe.named_steps['preprocessor']
                    .named_transformers_['cat']
                    .get_feature_names_out(cat_features))
all_feature_names = num_features + ohe_features

# Try to get feature importances
classifier = best_pipe.named_steps['classifier']
if hasattr(classifier, 'feature_importances_'):
    importances = classifier.feature_importances_
elif hasattr(classifier, 'coef_'):
    importances = np.abs(classifier.coef_[0])
else:
    importances = None

if importances is not None:
    feat_imp = pd.DataFrame({'feature': all_feature_names, 'importance': importances})
    feat_imp = feat_imp.sort_values('importance', ascending=True)

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.barh(feat_imp['feature'], feat_imp['importance'], color='#5B8DEF')
    ax.set_xlabel('Importance')
    ax.set_title(f'Feature Importance: {best_name}')
    plt.tight_layout()
    plt.show()
else:
    print('Feature importances not available for this model type.')

## 10. Export best model

In [ ]:
import os
os.makedirs('../models', exist_ok=True)

model_path = '../models/best_delay_model.joblib'
joblib.dump(best_pipe, model_path)
print(f'Saved best model ({best_name}) to {model_path}')

# Save feature config for downstream use
feature_config = {
    'num_features': num_features,
    'cat_features': cat_features,
    'target': target,
    'model_name': best_name,
    'test_auc': float(results_df.loc[best_name, 'AUC']),
    'baseline_auc': float(baseline_auc),
}

import json
with open('../models/feature_config.json', 'w') as f:
    json.dump(feature_config, f, indent=2)

print(f'Saved feature config to ../models/feature_config.json')
print(f'\nModel AUC: {feature_config["test_auc"]:.4f} vs Baseline AUC: {feature_config["baseline_auc"]:.4f}')

## Summary

| Model | AUC | Notes |
|-------|-----|-------|
| Baseline (historical freq) | ___ | Current web app approach |
| Logistic Regression | ___ | Simple linear model |
| Random Forest | ___ | Ensemble, handles non-linearity |
| Gradient Boosting | ___ | Usually best for tabular data |

_Fill in AUC values after running._

### Next steps
- If the best model significantly outperforms baseline, integrate into the web app
- Consider hyperparameter tuning (GridSearchCV / RandomizedSearchCV)
- Consider cross-validation for more robust estimates
- Investigate which routes/airports benefit most from the model vs baseline